# 🏋️ How2Sign — Optimized CSLR Training Pipeline

**Model:** BiLSTM (256) + Multi-Head Attention + BiLSTM (256) + CTC (`tf.nn.ctc_loss`)  
**Features:** 232-dim — body (50) + left-hand (42) + right-hand (42) + face (98)  
**Dataset:** `hadeelgamal/artifacts2` — pre-extracted `.npy` feature files  
**Sequence length:** Dynamic (P95 of real clip lengths, rounded to ×8)  
**Outputs:** `best_cslr.weights.h5`, `vocab.json`, `model_config.json`

In [ ]:
# ============================================================
# 1. SETUP & CONFIGURATION
# ============================================================
import os, glob, json, time
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path

try:
    import h5py
except ImportError:
    os.system('pip install -q h5py')
    import h5py

try:
    from tqdm.auto import tqdm
except ImportError:
    os.system('pip install -q tqdm')
    from tqdm.auto import tqdm

# --- GPU / mixed-precision setup ----------------------------------------
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')

# --- System info --------------------------------------------------------
print('=' * 55)
print(f'  TensorFlow : {tf.__version__}')
print(f'  NumPy      : {np.__version__}')
print(f'  Pandas     : {pd.__version__}')
if gpus:
    for i, g in enumerate(gpus):
        d = tf.config.experimental.get_device_details(g)
        print(f'  GPU {i}      : {d.get("device_name", g.name)}')
    print('  Precision  : mixed_float16')
else:
    print('  Mode       : CPU only')
try:
    import psutil
    r = psutil.virtual_memory()
    print(f'  RAM total  : {r.total/1e9:.1f} GB  avail: {r.available/1e9:.1f} GB')
except ImportError:
    pass
print('=' * 55)


## 2. Dataset Paths

Dataset: `hadeelgamal/artifacts2` — contains pre-extracted `.npy` feature files  
(232-dim per frame) and three subset CSVs with columns `SENTENCE_NAME`, `NPY_PATH`, `SENTENCE`.

In [ ]:
# ============================================================
# 2. DATASET PATHS
# ============================================================
BASE_DIR = Path('/kaggle/input/datasets/hadeelgamal/artifacts2')

CSV_TRAIN = BASE_DIR / 'how2sign_train_subset.csv'
CSV_VAL   = BASE_DIR / 'how2sign_val_subset.csv'
CSV_TEST  = BASE_DIR / 'how2sign_test_subset.csv'

FEAT_DIR_TRAIN = BASE_DIR / 'train_features'
FEAT_DIR_VAL   = BASE_DIR / 'val_features'
FEAT_DIR_TEST  = BASE_DIR / 'test_features'

OUTPUT_DIR   = Path('/kaggle/working')
H5_TRAIN     = OUTPUT_DIR / 'train_features.h5'
VOCAB_PATH   = OUTPUT_DIR / 'vocab.json'
WEIGHTS_PATH = OUTPUT_DIR / 'best_cslr.weights.h5'
CONFIG_PATH  = OUTPUT_DIR / 'model_config.json'

# Fixed hyper-parameters
NUM_FEATURES = 232     # body(50) + lhand(42) + rhand(42) + face(98)
MAX_LABEL    = 50      # max words per sentence
MAX_TOKENS   = 3000    # vocabulary cap
BATCH_SIZE   = 32
EPOCHS       = 30
# SEQUENCE_LENGTH is set dynamically in the next cell

print('Paths:')
for label, p in [('BASE_DIR',    BASE_DIR),
                  ('CSV_TRAIN',   CSV_TRAIN),
                  ('FEAT_TRAIN',  FEAT_DIR_TRAIN),
                  ('OUTPUT_DIR',  OUTPUT_DIR)]:
    print(f'  {label:12s} exists={p.exists()}  {p}')


## 3. Sequence Length Analysis

Scans the real `.npy` clip lengths and sets `SEQUENCE_LENGTH` to the **95th percentile** (rounded up to a multiple of 8 for clean GPU batching).  
This ensures >95% of clips are kept at full length with no arbitrary hard cap.

In [ ]:
# ============================================================
# 3. SEQUENCE LENGTH ANALYSIS  (auto-sets SEQUENCE_LENGTH)
# ============================================================
def analyse_lengths(feature_dirs):
    lengths = []
    for d in feature_dirs:
        for fp in glob.glob(os.path.join(d, '*.npy')):
            try:
                arr = np.load(fp, mmap_mode='r')
                lengths.append(arr.shape[0])
            except Exception:
                pass
    return np.array(lengths, dtype=np.int32)

lengths = analyse_lengths([FEAT_DIR_TRAIN, FEAT_DIR_VAL, FEAT_DIR_TEST])

if len(lengths) == 0:
    print('WARNING: No .npy files found. Defaulting SEQUENCE_LENGTH = 300.')
    SEQUENCE_LENGTH = 300
else:
    p50  = int(np.percentile(lengths, 50))
    p90  = int(np.percentile(lengths, 90))
    p95  = int(np.percentile(lengths, 95))
    p99  = int(np.percentile(lengths, 99))
    pmax = int(lengths.max())
    pmin = int(lengths.min())
    pmean= int(lengths.mean())

    print(f'Frame-length distribution across {len(lengths):,} clips:')
    print(f'  Min    : {pmin:>5} frames')
    print(f'  Mean   : {pmean:>5} frames')
    print(f'  P50    : {p50:>5} frames   (median)')
    print(f'  P90    : {p90:>5} frames')
    print(f'  P95    : {p95:>5} frames   <- chosen as SEQUENCE_LENGTH')
    print(f'  P99    : {p99:>5} frames')
    print(f'  Max    : {pmax:>5} frames')

    # Round up to nearest multiple of 8 for clean GPU batching
    SEQUENCE_LENGTH = int(np.ceil(p95 / 8) * 8)
    pct = (lengths <= SEQUENCE_LENGTH).mean() * 100
    print(f'\nSEQUENCE_LENGTH = {SEQUENCE_LENGTH}  '
          f'(P95 rounded to x8 — preserves {pct:.1f}% of clips fully)')

    if SEQUENCE_LENGTH < 200:
        print('WARNING: SEQUENCE_LENGTH < 200. Consider using P99 instead.')


## 4. HDF5 Data Packer

Packs all `.npy` files into a single compressed HDF5 file for fast I/O during training.  
**Runs once** — skips automatically if the file already exists.  
Gzip compression (level 1) gives good size reduction with very low CPU overhead.

In [ ]:
# ============================================================
# 4. HDF5 DATA PACKER  (run once — skips if already exists)
# ============================================================
if H5_TRAIN.exists():
    print(f'HDF5 already exists ({H5_TRAIN.stat().st_size/1e6:.0f} MB) — skipping.')
else:
    if CSV_TRAIN.exists():
        print('Packing train .npy files into HDF5 for fast I/O…')
        df = pd.read_csv(CSV_TRAIN)
        packed = 0
        with h5py.File(H5_TRAIN, 'w') as hf:
            for _, row in tqdm(df.iterrows(), total=len(df), desc='Packing'):
                name     = str(row.get('SENTENCE_NAME', ''))
                npy_path = BASE_DIR / str(row.get('NPY_PATH', ''))
                if npy_path.exists():
                    data = np.load(npy_path)
                    hf.create_dataset(name, data=data,
                                      compression='gzip', compression_opts=1)
                    packed += 1
        print(f'Packed {packed:,} clips -> {H5_TRAIN}  '
              f'({H5_TRAIN.stat().st_size/1e6:.0f} MB)')
    else:
        print(f'CSV not found: {CSV_TRAIN} — skipping HDF5 pack.')


## 5. Tokenizer

Builds a word vocabulary from training sentences.  
Index **0 is reserved for the CTC blank token** — all word indices are shifted +1.

In [ ]:
# ============================================================
# 5. TOKENIZER
# ============================================================
from tensorflow.keras.layers import TextVectorization

def _read_csv(path):
    df = pd.read_csv(path, sep='\t', on_bad_lines='skip')
    if 'SENTENCE' not in df.columns:
        df = pd.read_csv(path, on_bad_lines='skip')
    return df

train_df = _read_csv(CSV_TRAIN)
val_df   = _read_csv(CSV_VAL)
test_df  = _read_csv(CSV_TEST)

train_sentences = train_df['SENTENCE'].fillna('').tolist()
val_sentences   = val_df['SENTENCE'].fillna('').tolist()
test_sentences  = test_df['SENTENCE'].fillna('').tolist()

tokenizer = TextVectorization(
    max_tokens=MAX_TOKENS,
    standardize='lower_and_strip_punctuation',
    split='whitespace',
    output_mode='int'
)
print('Adapting tokenizer…')
tokenizer.adapt(train_sentences)
vocab = tokenizer.get_vocabulary()

# Index 0 reserved for CTC blank
def encode_sentence(text: str) -> np.ndarray:
    enc = tokenizer([text])[0].numpy() + 1
    enc = np.clip(enc, 1, MAX_TOKENS)  # safety clamp
    return enc

def decode_indices(indices) -> str:
    words = []
    for idx in indices:
        if idx == 0:
            continue
        w_idx = int(idx) - 1
        if 0 <= w_idx < len(vocab):
            w = vocab[w_idx]
            if w not in ('', '[UNK]'):
                words.append(w)
    return ' '.join(words)

# Save vocab for inference notebook
with open(VOCAB_PATH, 'w') as f:
    json.dump(vocab, f)
print(f'Vocab saved -> {VOCAB_PATH}')

# Metrics
vocab_set = set(vocab)
oov_val   = {w for s in val_sentences  for w in s.lower().split()} - vocab_set
oov_test  = {w for s in test_sentences for w in s.lower().split()} - vocab_set
lens_tr   = [len(s.split()) for s in train_sentences]
lens_va   = [len(s.split()) for s in val_sentences]
lens_te   = [len(s.split()) for s in test_sentences]

print('\n' + '=' * 55)
print(f'  Vocabulary size         : {len(vocab):,}')
print(f'  Top-10 tokens           : {vocab[:10]}')
print(f'  OOV (val / test)        : {len(oov_val):,} / {len(oov_test):,}')
print(f'  Sentence len  (train)   : min={min(lens_tr)} max={max(lens_tr)} '
      f'mean={np.mean(lens_tr):.1f} std={np.std(lens_tr):.1f}')
print('=' * 55)

# Smoke test
s0  = train_sentences[0] if train_sentences else 'hello world'
enc = encode_sentence(s0)
print(f'\nSample  : "{s0}"')
print(f'Encoded : {enc[:8]} …')
print(f'Decoded : "{decode_indices(enc)}"')


## 6. CTC Data Generator

Reads from HDF5 (fast) or falls back to individual `.npy` files.  
Augmentation (train only): spatial jitter N(0, 0.005) + scale jitter ×[0.95, 1.05].  
CTC safety guard: `label_length` is clamped to `input_length` to prevent NaN loss.

In [ ]:
# ============================================================
# 6. CTC DATA GENERATOR
# ============================================================
from tensorflow.keras.utils import Sequence

class How2SignCTCGenerator(Sequence):
    COL_NAME     = 'SENTENCE_NAME'
    COL_NPY      = 'NPY_PATH'
    COL_SENTENCE = 'SENTENCE'

    def __init__(self, csv_path: Path, base_path: Path,
                 h5_path: Path         = None,
                 batch_size: int       = BATCH_SIZE,
                 sequence_length: int  = None,
                 max_label_len: int    = MAX_LABEL,
                 num_features: int     = NUM_FEATURES,
                 augment: bool         = False):

        self.base_path       = base_path
        self.batch_size      = batch_size
        self.sequence_length = sequence_length or SEQUENCE_LENGTH
        self.max_label_len   = max_label_len
        self.num_features    = num_features
        self.augment         = augment

        # Load CSV
        if csv_path and Path(csv_path).exists():
            df = pd.read_csv(csv_path, sep='\t', on_bad_lines='skip')
            if self.COL_NAME not in df.columns:
                df = pd.read_csv(csv_path, on_bad_lines='skip')
            # Keep only rows with a valid NPY_PATH
            if self.COL_NPY in df.columns:
                df = df[df[self.COL_NPY].notna()].reset_index(drop=True)
            self.df = df
        else:
            self.df = pd.DataFrame()

        # Open HDF5 if available
        self._h5 = None
        if h5_path and Path(h5_path).exists():
            self._h5 = h5py.File(h5_path, 'r')

        h5_keys = len(self._h5) if self._h5 else 0
        cov     = 100.0 * h5_keys / max(len(self.df), 1)
        print(f'  {Path(csv_path).name if csv_path else "?"}: '
              f'{len(self.df):,} rows | H5 keys={h5_keys:,} ({cov:.0f}%) | '
              f'augment={augment}')

    def __len__(self):
        return max(1, int(np.ceil(len(self.df) / self.batch_size)))

    def _load(self, row) -> np.ndarray:
        name = str(row.get(self.COL_NAME, ''))
        # Try HDF5 first
        if self._h5 and name in self._h5:
            return self._h5[name][:]
        # Fall back to .npy file via NPY_PATH column
        npy_rel = str(row.get(self.COL_NPY, ''))
        npy_abs = BASE_DIR / npy_rel
        if npy_abs.exists():
            return np.load(npy_abs)
        return np.zeros((1, self.num_features), dtype=np.float32)

    def __getitem__(self, idx):
        batch = self.df.iloc[idx * self.batch_size : (idx + 1) * self.batch_size]
        B     = len(batch)

        X             = np.zeros((B, self.sequence_length, self.num_features), dtype=np.float32)
        Y             = np.zeros((B, self.max_label_len),                      dtype=np.int32)
        input_lengths = np.zeros((B, 1), dtype=np.int32)
        label_lengths = np.zeros((B, 1), dtype=np.int32)

        for i, (_, row) in enumerate(batch.iterrows()):
            frames = self._load(row)
            T      = min(len(frames), self.sequence_length)
            T      = max(T, 1)

            data = frames[:T].astype(np.float32)
            if self.augment:
                data = data + np.random.normal(0, 0.005, data.shape).astype(np.float32)
                data = data * np.float32(np.random.uniform(0.95, 1.05))
            X[i, :T, :] = data

            sentence = str(row.get(self.COL_SENTENCE, ''))
            encoded  = encode_sentence(sentence)
            L = min(len(encoded), self.max_label_len)
            # CTC constraint: label_length <= input_length
            if L > T:
                L = T
            if L > 0:
                Y[i, :L] = np.clip(encoded[:L], 1, MAX_TOKENS)
            L = max(L, 1)  # avoid 0-length labels crashing CTC

            input_lengths[i, 0] = max(T, L + 1)  # input_len > label_len for CTC
            label_lengths[i, 0] = L

        return ({'input': X, 'labels': Y,
                 'input_length': input_lengths, 'label_length': label_lengths},
                np.zeros((B,), dtype=np.float32))

    def on_epoch_end(self):
        self.df = self.df.sample(frac=1).reset_index(drop=True)


print('Creating generators…')
train_gen = How2SignCTCGenerator(
    CSV_TRAIN, FEAT_DIR_TRAIN, h5_path=H5_TRAIN, augment=True)
val_gen   = How2SignCTCGenerator(
    CSV_VAL,   FEAT_DIR_VAL,   augment=False)
test_gen  = How2SignCTCGenerator(
    CSV_TEST,  FEAT_DIR_TEST,  augment=False)

print(f'\nBatches — train: {len(train_gen)} | val: {len(val_gen)} | test: {len(test_gen)}')

if len(train_gen) > 0:
    sb, _ = train_gen[0]
    print('\nSample batch shapes:')
    for k, v in sb.items():
        print(f'  {k:14s}: {v.shape}  dtype={v.dtype}')
    x = sb['input']
    print('\nFeature region activity (mean abs value):')
    print(f'  Body  [ 0: 50] : {np.abs(x[:,:, 0:50]).mean():.4f}')
    print(f'  LHand [50: 92] : {np.abs(x[:,:,50:92]).mean():.4f}')
    print(f'  RHand [92:134] : {np.abs(x[:,:,92:134]).mean():.4f}')
    print(f'  Face  [134:232]: {np.abs(x[:,:,134:232]).mean():.4f}')


## 7. Model Architecture — BiLSTM + Multi-Head Attention + CTC

```
Input (SEQUENCE_LENGTH, 232)
  → BiLSTM(256) → BatchNorm
  → MultiHeadAttention(4 heads, key_dim=64) + residual + LayerNorm
  → Dropout(0.3)
  → BiLSTM(256) → BatchNorm → Dropout(0.3)
  → Dense(vocab+1)  → Softmax
  → CTCLossLayer  [tf.nn.ctc_loss — Keras 3 compatible]
```

**CTC blank index = 0.** Uses `tf.nn.ctc_loss` (log-probs, time-major) with non-finite loss filtering.

In [ ]:
# ============================================================
# 7. MODEL ARCHITECTURE
# ============================================================
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization,
    LSTM, Bidirectional, Softmax,
    MultiHeadAttention, LayerNormalization
)
from tensorflow.keras.models import Model

VOCAB_SIZE_CTC = MAX_TOKENS + 1  # +1 for CTC blank at index 0
print(f'CTC output classes (incl. blank): {VOCAB_SIZE_CTC}')


class CTCLossLayer(tf.keras.layers.Layer):
    """
    Keras 3 / TF2 compatible CTC loss layer.
    Uses tf.nn.ctc_loss (log-softmax, time-major) for numerical stability.
    Non-finite losses are zeroed out to prevent NaN propagation.
    """
    def __init__(self, blank_index=0, **kwargs):
        super().__init__(**kwargs)
        self.blank_index = blank_index

    def call(self, inputs):
        y_pred, labels, input_length, label_length = inputs

        # log-softmax for numerical stability
        log_probs = tf.math.log(y_pred + 1e-8)

        # tf.nn.ctc_loss expects [T, B, C] (time-major)
        log_probs_tm = tf.transpose(log_probs, [1, 0, 2])

        input_len_1d = tf.cast(tf.reshape(input_length, [-1]), tf.int32)
        label_len_1d = tf.cast(tf.reshape(label_length, [-1]), tf.int32)

        # Safety clamps
        label_len_1d = tf.minimum(label_len_1d, input_len_1d)
        label_len_1d = tf.maximum(label_len_1d, 1)

        loss = tf.nn.ctc_loss(
            labels       = labels,
            logits       = log_probs_tm,
            label_length = label_len_1d,
            logit_length = input_len_1d,
            logits_time_major = True,
            blank_index  = self.blank_index,
        )

        # Filter non-finite losses
        loss = tf.where(tf.math.is_finite(loss), loss, tf.zeros_like(loss))
        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'blank_index': self.blank_index})
        return cfg


def build_model(seq_len, num_features, vocab_size_ctc, max_label):
    inp          = Input(shape=(seq_len, num_features), name='input')
    labels_in    = Input(shape=(None,), dtype='int32',  name='labels')
    input_len_in = Input(shape=(1,),    dtype='int32',  name='input_length')
    label_len_in = Input(shape=(1,),    dtype='int32',  name='label_length')

    # Encoder
    x1   = Bidirectional(LSTM(256, return_sequences=True), name='bilstm_1')(inp)
    x1   = BatchNormalization(name='bn_1')(x1)

    # Multi-Head Self-Attention with residual connection
    attn = MultiHeadAttention(num_heads=4, key_dim=64, name='mha')(x1, x1)
    x    = LayerNormalization(name='ln_1')(x1 + attn)
    x    = Dropout(0.3, name='drop_1')(x)

    x    = Bidirectional(LSTM(256, return_sequences=True), name='bilstm_2')(x)
    x    = BatchNormalization(name='bn_2')(x)
    x    = Dropout(0.3, name='drop_2')(x)

    # Dense logits then Softmax as a layer (not activation=)
    logits = Dense(vocab_size_ctc, name='logits')(x)
    y_pred = Softmax(name='prediction')(logits)

    # CTC loss
    ctc_out = CTCLossLayer(blank_index=0, name='ctc_loss')(
        [y_pred, labels_in, input_len_in, label_len_in])

    model_train = Model(
        inputs  = [inp, labels_in, input_len_in, label_len_in],
        outputs = ctc_out, name='ctc_train')
    model_train.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4, clipnorm=5.0))

    model_infer = Model(inputs=inp, outputs=y_pred, name='ctc_inference')
    return model_train, model_infer


model_train, model_infer = build_model(
    SEQUENCE_LENGTH, NUM_FEATURES, VOCAB_SIZE_CTC, MAX_LABEL)

model_train.summary()
model_infer.summary()
total_params     = model_train.count_params()
trainable_params = sum(tf.size(w).numpy() for w in model_train.trainable_weights)
print(f'\nTotal params     : {total_params:,}')
print(f'Trainable params : {trainable_params:,}')


## 8. Training

| Callback | Setting |
|---|---|
| EarlyStopping | patience=7, restore best weights |
| ReduceLROnPlateau | factor=0.5, patience=3, min_lr=1e-6 |
| ModelCheckpoint | saves full model (not weights-only) |

In [ ]:
# ============================================================
# 8. TRAINING LOOP
# ============================================================
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=7,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(str(WEIGHTS_PATH), monitor='val_loss',
                    save_best_only=True, save_weights_only=False, verbose=1),
]

history = model_train.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
)

# --- Summary printout ---------------------------------------------------
h     = history.history
best  = int(np.argmin(h['val_loss']))
print('\n' + '=' * 55)
print(f'  Epochs ran       : {len(h["loss"])}')
print(f'  Best epoch       : {best + 1}')
print(f'  Best val_loss    : {h["val_loss"][best]:.4f}')
print(f'  Final train_loss : {h["loss"][-1]:.4f}')
print(f'  Final val_loss   : {h["val_loss"][-1]:.4f}')
print('=' * 55)

# --- Training curves plot -----------------------------------------------
epochs_x = range(1, len(h['loss']) + 1)
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(epochs_x, h['loss'],     'b-o', markersize=4, label='Train CTC Loss')
ax.plot(epochs_x, h['val_loss'], 'r-o', markersize=4, label='Val CTC Loss')
ax.axvline(best + 1, color='green', linestyle='--', alpha=0.7,
           label=f'Best epoch ({best+1})')
ax.set_xlabel('Epoch')
ax.set_ylabel('CTC Loss')
ax.set_title('Training & Validation CTC Loss')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUTPUT_DIR / 'training_curves.png'), dpi=150)
plt.show()
print(f'Plot saved -> {OUTPUT_DIR}/training_curves.png')


## 9. Save Config & Package for Download

Saves model config for the inference notebook, then zips everything into one archive for easy download from the Kaggle sidebar.

In [ ]:
# ============================================================
# 9. SAVE MODEL CONFIG FOR INFERENCE NOTEBOOK
# ============================================================
h = history.history
best = int(np.argmin(h['val_loss']))

config = {
    'seq_len'         : SEQUENCE_LENGTH,
    'num_features'    : NUM_FEATURES,
    'max_label'       : MAX_LABEL,
    'vocab_size_ctc'  : VOCAB_SIZE_CTC,
    'max_tokens'      : MAX_TOKENS,
    'batch_size'      : BATCH_SIZE,
    'best_epoch'      : best + 1,
    'best_val_loss'   : float(h['val_loss'][best]),
}
with open(CONFIG_PATH, 'w') as f:
    json.dump(config, f, indent=2)

print('Outputs:')
for p in [WEIGHTS_PATH, VOCAB_PATH, CONFIG_PATH,
          OUTPUT_DIR / 'training_curves.png']:
    if Path(p).exists():
        print(f'  {Path(p).name:40s} {Path(p).stat().st_size/1e6:.1f} MB')


In [ ]:
# ============================================================
# 10. PACKAGE EVERYTHING FOR KAGGLE DOWNLOAD
# ============================================================
import zipfile

OUTPUT_ZIP = OUTPUT_DIR / 'how2sign_cslr_artifacts.zip'

def zip_folder(folder, zf, prefix):
    for fpath in sorted(glob.glob(os.path.join(folder, '**', '*'), recursive=True)):
        if os.path.isfile(fpath):
            zf.write(fpath, prefix + '/' + os.path.relpath(fpath, folder))

print(f'Building archive -> {OUTPUT_ZIP}')
with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as zf:

    # Model files
    for fname in ['best_cslr.weights.h5', 'vocab.json',
                  'model_config.json',    'training_curves.png']:
        p = OUTPUT_DIR / fname
        if p.exists():
            zf.write(str(p), fname)
            print(f'  {fname}')

zip_mb = OUTPUT_ZIP.stat().st_size / 1e6
print(f'\nDone! Archive: {OUTPUT_ZIP}  ({zip_mb:.1f} MB)')
print('Kaggle sidebar -> Output -> how2sign_cslr_artifacts.zip -> Download')
